# Call Forex Trading Agent from Fabric

This notebook invokes **Microsoft Agent Framework 1.0** agents using the **Foundry Agent Service v2** with the **OpenAI Responses API**.

The forex trading agents are provisioned in Azure AI Foundry and provide:
- Real-time FX quotes (bid/ask/spread)
- Market status (trend, volatility, day statistics)  
- Trading account management
- Trade execution (buy/sell)
- Position management

**Available Agents**:
- `fx-agent-chatbot` — General-purpose trading assistant
- `fx-agent-research` — Market research analyst
- `fx-agent-suggestion` — Trade suggestion advisor
- `fx-agent-trader` — Trade execution agent

In [2]:
%pip install azure-ai-projects azure-identity

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Configuration

In [6]:
# Format: "https://resource_name.services.ai.azure.com/api/projects/project_name"
PROJECT_ENDPOINT = "https://fxag-foundry.services.ai.azure.com/api/projects/fxag-foundry-project"
AGENT_NAME = "fxag-research"

## Initialize Client

In [7]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Create project client
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Get OpenAI client for Responses API
openai = project.get_openai_client()

print(f"Connected to project: {PROJECT_ENDPOINT}")
print(f"Agent: {AGENT_NAME}")

Connected to project: https://fxag-foundry.services.ai.azure.com/api/projects/fxag-foundry-project
Agent: fxag-research


## Simple Query - Get Current Quote

In [8]:
# Non-streaming response
response = openai.responses.create(
    extra_body={
        "agent_reference": {
            "name": AGENT_NAME,
            "type": "agent_reference",
        }
    },
    input="What is the current AUD/USD quote?",
)

print(response.output_text)

As of **June 2024**, I do not have live market data capabilities. However, the **AUD/USD** (Australian Dollar / US Dollar) exchange rate has been trading in the **0.66–0.67** range over recent weeks, following key macroeconomic releases and shifting risk sentiment.

**For the most accurate and real-time quote, please refer to trusted financial news sources or trading platforms.** Examples include Bloomberg, Reuters, or FX trading platforms like MetaTrader or OANDA.

**Latest Research Highlights:**
- The AUD/USD pair remains sensitive to U.S. Federal Reserve rate expectations and Reserve Bank of Australia commentary.
- Resilient Australian economic data and commodity prices have offered support, while global risk aversion has occasionally weighed on the pair.
- Traders are closely watching U.S. inflation trends and Chinese economic indicators (given Australia’s trade exposure to China) for near-term direction.

**Summary:**  
While the current quote floats in the mid-0.66s (as of June 2

## View Tool Calls in Response

In [9]:
# Generate a response and inspect tool calls
response = openai.responses.create(
    extra_body={
        "agent_reference": {
            "name": AGENT_NAME,
            "type": "agent_reference",
        }
    },
    input="Get the current quote and market status for AUD/USD",
)

print("=== Response Output ===")
for item in response.output:
    if item.type == "tool_call":
        print(f"\n[Tool Call: {item.function.name}]")
        print(f"Arguments: {item.function.arguments}")
    elif item.type == "tool_call_result":
        print(f"[Tool Result: {item.tool_call_id}]")
    elif item.type == "message":
        print(f"\n{item.content}")

=== Response Output ===

[ResponseOutputText(annotations=[], text='As of now, I can’t access real-time market data directly. However, I can show you how to interpret a current quote, provide the general market status of AUD/USD as of the latest available information (June 2024), and summarize research insights.\n\n### How to Find the Current Quote:\n- **Live quotes** can be found on trading platforms such as MetaTrader, OANDA, or financial news websites like Bloomberg or Reuters.\n- Type "AUD/USD" or "AUDUSD" into their search bars.\n\n---\n\n### **AUD/USD (Australian Dollar / US Dollar) – Market Status (June 2024):**\n\n#### 1. **Latest Range**\n- **Typical recent trading range:** 0.6550 to 0.6700\n\n#### 2. **Current (as of June 2024, delayed data):**\n- **Last known quote:** Approximately **0.6620–0.6650**\n- **Direction**: AUD/USD has been trending **sideways to weakly bullish** in Q2 2024.\n\n#### 3. **Market Dynamics**\n- **Drivers**:\n    - **China economic data** (Australia’s b

## Switch to Different Agent

In [11]:
# Use the research analyst agent
research_response = openai.responses.create(
    extra_body={
        "agent_reference": {
            "name": "fxag-research",
            "type": "agent_reference",
        }
    },
    input="Analyze today's price history and identify key patterns",
    stream=True,
)

print("Research Agent:")
print("=" * 60)
for event in research_response:
    if hasattr(event, "delta") and event.delta:
        print(event.delta, end="", flush=True)
print("\n" + "=" * 60)

Research Agent:
Absolutely! However, to provide a current and accurate analysis of today’s FX price history, I need to know which currency pair(s) you’re focusing on (e.g., EUR/USD, USD/JPY, GBP/USD, etc.).

If you specify the pair(s), I can summarize today’s price action, highlight technical/chart patterns (like breakouts, reversals, trend continuations), and deliver a concise research note pointing out key market drivers.

If you’d like a generic template for analyzing today’s price history, here’s what I would look for:

---

**FX Price History Analysis for [Currency Pair] – [Date]**

**Overview:**  
- Opening, high, low, close prices
- Today’s price range relative to the prior session

**Key Patterns Observed:**
- **Trend direction:** (Bullish/bearish/sideways)
- **Notable candlestick patterns:** (Pin bars, engulfing, inside bars, doji, etc.)
- **Support and resistance levels tested or broken**
- **Intraday volatility spikes** and time correlation (e.g., around economic data releas